# Coco Crepe — 03 Gold Product Master

Publicación del Data Product maestro de productos.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
products_clean = f"{PRODUCT_CATALOG}.silver.products_clean"
product_master = f"{PRODUCT_CATALOG}.gold.product_master" 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {product_master} AS
SELECT
    product_id,
    product_name,
    category,
    price,
    TRUE AS is_active,
    current_timestamp() AS published_at
FROM {products_clean}
""")

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    COUNT(DISTINCT product_id) AS distinct_products,
    MIN(price) AS minimum_price,
    MAX(price) AS maximum_price
FROM {product_master}
""").display()

spark.sql(
    f"SELECT * FROM {product_master} ORDER BY product_id LIMIT 20"
).display()